In [1]:
import os
os.makedirs("results", exist_ok=True)

import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

expression = pd.read_csv("../assignment3/shared_expression_log2.csv", index_col=0)
sample_meta = pd.read_csv("../assignment3/shared_sample_metadata.csv", index_col=0)
kmeans = pd.read_csv("../assignment3/results/kmeans_cluster_assignments.csv", index_col=0)
assert (expression.index == sample_meta.index).all() and (kmeans.index == sample_meta.index).all()

In [2]:
ranked = expression.var(axis=0).sort_values(ascending=False).index

strat = sample_meta["condition"].astype(str) + "*" + sample_meta["age"].astype(str) + "*" + sample_meta["genotype"].astype(str)
folds = pd.Series(-1, index=sample_meta.index, name="fold")
for i, (_, test_idx) in enumerate(StratifiedKFold(4, shuffle=True, random_state=42).split(expression, strat)):
    folds.iloc[test_idx] = i

print(folds.value_counts())
folds.to_csv("results/cv_folds.csv")
cv = [(np.where(folds != i)[0], np.where(folds == i)[0]) for i in range(4)]

fold
1    8
3    8
0    8
2    8
Name: count, dtype: int64


In [3]:
y_cond = (sample_meta["condition"] == "flight").astype(int)
y_clust = (kmeans["kmeans_k2"] == sorted(kmeans["kmeans_k2"].unique())[1]).astype(int)
targets = {"condition": y_cond, "cluster_k2": y_clust}

In [4]:
def run_logreg(n_genes, y):
    X = expression[ranked[:n_genes]]
    model = make_pipeline(StandardScaler(), LogisticRegression(solver="liblinear", max_iter=5000))
    proba = cross_val_predict(model, X, y, cv=cv, method="predict_proba")[:, 1]
    pred = (proba >= 0.5).astype(int)
    auc = roc_auc_score(y, proba)
    return pred, proba, auc

rows, preds, scores = [], {}, {}
for n in [10, 100, 1000, 5000, 10000]:
    for name, y in targets.items():
        pred, proba, auc = run_logreg(n, y)
        rows.append({"Method": "Logistic Regression", "Genes": n, "Target": name,
                     "AUC": auc, "Accuracy": (pred == y.values).mean()})
        preds[f"logreg_{name}_{n}genes"] = pred
        scores[f"logreg_{name}_{n}genes"] = proba

auc_table = pd.DataFrame(rows)
auc_table.to_csv("results/logreg_auc_by_gene_count.csv", index=False)
pd.DataFrame(preds, index=expression.index).to_csv("results/logreg_predictions.csv")
pd.DataFrame(scores, index=expression.index).to_csv("results/logreg_probabilities.csv")
auc_table

,Method,Genes,Target,AUC,Accuracy
0,Logistic Regression,10,condition,0.652344,0.62500
1,Logistic Regression,10,cluster_k2,1.000000,1.00000
2,Logistic Regression,100,condition,0.914062,0.90625
3,Logistic Regression,100,cluster_k2,1.000000,1.00000
4,Logistic Regression,1000,condition,1.000000,1.00000
5,Logistic Regression,1000,cluster_k2,1.000000,1.00000
6,Logistic Regression,5000,condition,1.000000,1.00000
7,Logistic Regression,5000,cluster_k2,1.000000,1.00000
8,Logistic Regression,10000,condition,1.000000,1.00000
9,Logistic Regression,10000,cluster_k2,1.000000,1.00000
